# 01 - Explorer et nettoyer les données de l'or (XAU/USD)

**Objectif :** regarder le dataset de près, comprendre ses défauts et son comportement, puis produire un **dataset propre** pour la suite (features, modèles).

**Comment l'utiliser :**
- Exécute les cellules une par une avec **Maj + Entrée** et lis les commentaires avant chaque résultat.
- Chaque partie se termine par un encadré **« À retenir »** : c'est ce qu'on écrira dans le rapport.
- Les décisions de nettoyage sont des **variables en haut de la partie 11** : tu peux les changer et relancer.

**Plan :**
1. Préparation
2. Premier regard sur les données
3. Vue d'ensemble : le prix dans le temps
4. Calendrier : jours manquants
5. Cohérence des bougies OHLC
6. Rendements : distribution et queues épaisses
7. Valeurs extrêmes : vrais chocs ou erreurs ?
8. Volatilité et régimes
9. Relations entre variables (corrélations, autocorrélation, saisonnalité)
10. Stationnarité
11. Nettoyage pas à pas
12. Vérifications finales et sauvegarde

> **Règle d'or (anti-fuite) :** tous les seuils et toutes les statistiques qui servent à *décider* sont calculés sur le **train seulement** (2004-2018). La validation et le test, on les regarde pour vérifier la qualité des données, jamais pour régler quelque chose.

## 1. Préparation

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Les chemins de config.yaml sont relatifs à la racine du dépôt.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from gold_agents.config import load_config
from gold_agents.data.load import read_kaggle_csv, read_update_csv
from gold_agents.eda import analysis

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})

BLEU, ORANGE, VERT, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
COULEURS = {"train": BLEU, "val": ORANGE, "test": VERT}

cfg = load_config()
SPLITS = cfg.splits
print(
    "Train jusqu'au",
    SPLITS.train_end,
    "| validation",
    SPLITS.val_start,
    "->",
    SPLITS.val_end,
    "| test à partir du",
    SPLITS.test_start,
)

## 2. Premier regard sur les données

On lit le CSV Kaggle (historique figé, la seule source pour train/val/test) et le fichier de mise à jour Twelve Data (jours après la fin du CSV Kaggle, gardés à part).

In [ ]:
kaggle = read_kaggle_csv(cfg.data.raw_path)
update = (
    read_update_csv(cfg.data.update_path) if cfg.data.update_path.exists() else kaggle.iloc[0:0]
)
update = update[update["date"] > kaggle["date"].max()].reset_index(drop=True)

debut, fin = kaggle["date"].min(), kaggle["date"].max()
print(f"Kaggle : {len(kaggle)} lignes, du {debut:%Y-%m-%d} au {fin:%Y-%m-%d}")
print(f"Mise à jour : {len(update)} lignes")
kaggle.head()

In [ ]:
# Les dernières lignes, et les types de chaque colonne
display(kaggle.tail())
kaggle.dtypes

In [ ]:
# Statistiques de base : min, max, moyenne de chaque colonne
kaggle.describe().round(2)

In [ ]:
# Doublons et valeurs manquantes.
# Le chargeur garde déjà une seule ligne par date :
# on vérifie dans le fichier brut combien il y en avait.
brut = pd.read_csv(cfg.data.raw_path, sep=None, engine="python")
print("Lignes dans le fichier brut :", len(brut))
print("Dates en double dans le brut :", brut.iloc[:, 0].duplicated().sum())
print("Valeurs manquantes par colonne :")
kaggle.isna().sum()

> **À retenir :** une ligne = un jour de cotation, avec `open`, `high`, `low`, `close` en dollars par once et un `volume`. Dans MetaTrader 4, ce volume est un **nombre de ticks** (cotations reçues par le courtier), pas un volume échangé.

## 3. Vue d'ensemble : le prix dans le temps

Échelle **logarithmique** : une même hauteur verticale = un même pourcentage de variation. C'est plus juste qu'une échelle normale quand le prix est passé de 400 à plus de 4 000 dollars.

In [ ]:
prix = kaggle.set_index("date")["close"]
ensemble = analysis.assign_split(prix.index, SPLITS)

fig, ax = plt.subplots()
for nom, couleur in COULEURS.items():
    partie = prix[ensemble.to_numpy() == nom]
    ax.plot(partie.index, partie, color=couleur, lw=1, label=nom)
if not update.empty:
    ax.plot(update["date"], update["close"], color=GRIS, lw=1, label="mise à jour API")
ax.set_yscale("log")
ax.set_title("Clôture XAU/USD (échelle log) et découpage train / validation / test")
ax.set_ylabel("USD par once")
ax.legend()
plt.show()

In [ ]:
# Combien de jours et quelle performance dans chaque ensemble ?
apercu = analysis.split_preview(kaggle, SPLITS, cfg.features.window)
pd.DataFrame(apercu["splits"]).T[
    [
        "first_date",
        "last_date",
        "n_rows",
        "n_usable_after_purge",
        "annualized_volatility",
        "buy_and_hold_return",
    ]
]

> **À retenir :** 67 % train / 14 % validation / 19 % test. Le test est un marché très haussier : une stratégie qui sort souvent du marché aura du mal à battre le simple « acheter et garder ». Il faudra aussi comparer le **risque**, pas seulement le profit.

## 4. Calendrier : jours manquants

L'or ne cote pas le week-end ni certains jours fériés (Noël, Nouvel An, Vendredi saint). On compare les dates présentes à tous les jours ouvrés (lundi-vendredi).

In [ ]:
cal = analysis.calendar_report(kaggle)
print("Lignes par jour de la semaine :", cal["rows_per_weekday"])
print("Jours ouvrés absents :", cal["missing_business_days"])

par_an = pd.Series(cal["missing_business_days_per_year"])
ax = par_an.plot.bar(color=BLEU, title="Jours ouvrés absents par année (3-4 = fériés normaux)")
ax.axhline(4, color=ORANGE, ls="--", lw=1)
plt.show()

pd.DataFrame(cal["largest_gaps"])

Deux trous sortent du lot en **2025** et **2026** : ce ne sont pas des jours fériés, l'export Kaggle n'a simplement pas enregistré ces jours. Zoomons dessus.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fenetres = [("2025-08-01", "2025-11-15"), ("2025-12-20", "2026-01-30")]
for ax, (debut, fin) in zip(axes, fenetres, strict=True):
    zoom = kaggle[(kaggle["date"] >= debut) & (kaggle["date"] <= fin)]
    ax.plot(zoom["date"], zoom["close"], marker="o", ms=3, color=VERT)
    ax.set_title(f"Zoom {debut} -> {fin}")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Les trous : la courbe « saute » d'un coup, faute de points entre les deux")
plt.tight_layout()
plt.show()

r = analysis.log_returns(kaggle)
print("Rendement affiché le jour qui suit chaque trou :")
for g in cal["largest_gaps"][:2]:
    print(
        f"  {g['before']} : {r[pd.Timestamp(g['before'], tz='UTC')] * 100:+.2f} %  "
        f"(cumul de {g['missing_business_days'] + 1} jours ouvrés, pas d'un seul)"
    )

> **À retenir :** les données sont complètes sauf **deux trous dans le test** (22 jours ouvrés en septembre-octobre 2025, 6 jours en janvier 2026). Le « rendement d'un jour » qui suit chaque trou est faux : il cumule plusieurs semaines. On le traite à la partie 11.

## 5. Cohérence des bougies OHLC

Une bougie est cohérente si `low ≤ open ≤ high`, `low ≤ close ≤ high` et si tous les prix sont strictement positifs.

In [ ]:
ohlc = analysis.ohlc_report(kaggle)
pd.Series({k: v for k, v in ohlc.items() if k != "violation_examples"})

In [ ]:
# L'amplitude (high - low) / close : les journées les plus agitées
amplitude = kaggle.set_index("date").eval("(high - low) / close") * 100
ax = amplitude.plot(color=BLEU, lw=0.6, title="Amplitude quotidienne (high - low) / close, en %")
plt.show()
amplitude.nlargest(5).round(2)

> **À retenir :** aucune bougie incohérente, aucun prix nul ou manquant. Les plus grandes amplitudes tombent en 2008, 2013 et 2020 : ce sont de vraies crises.

## 6. Rendements : distribution et queues épaisses

Un modèle ne prédit pas le prix mais le **rendement logarithmique** : `r(t) = ln(close_t / close_{t-1})`. Pour une petite variation, c'est presque égal au pourcentage de variation.

In [ ]:
r = analysis.log_returns(kaggle)
r_train = analysis.train_part(r, SPLITS)

ax = (r * 100).plot(color=BLEU, lw=0.5, title="Rendement quotidien (%)")
ax.axhline(0, color=GRIS, lw=0.8)
plt.show()

stats = analysis.returns_report(r_train)
pd.Series({k: v for k, v in stats.items() if not isinstance(v, dict)}).round(5)

In [ ]:
from scipy import stats as st

x = r_train * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme comparé à une loi normale (échelle log : on voit les queues)
axes[0].hist(x, bins=120, density=True, color=BLEU, alpha=0.8)
grille = np.linspace(x.min(), x.max(), 400)
normale = st.norm.pdf(grille, x.mean(), x.std())
axes[0].plot(grille, normale, color=ORANGE, lw=2, label="loi normale")
axes[0].set_yscale("log")
axes[0].set_ylim(1e-4)
axes[0].set_title("Distribution (train) vs loi normale")
axes[0].legend()

# QQ-plot : si les rendements étaient normaux, les points suivraient la droite
st.probplot(x, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot : les extrémités s'écartent de la droite")
plt.tight_layout()
plt.show()

> **À retenir :**
> - **Queues épaisses** : les journées extrêmes (au-delà de 4 écarts-types) sont environ 70 fois plus fréquentes qu'avec une loi normale. → On utilisera la **perte de Huber** et le **RobustScaler**, moins sensibles à ces jours que la MSE et le StandardScaler.
> - **Asymétrie négative** : les fortes baisses sont un peu plus violentes que les fortes hausses.

## 7. Valeurs extrêmes : vrais chocs ou erreurs ?

On mesure l'écart de chaque jour à la normale avec un **z-score robuste** : (valeur - médiane) / MAD. La médiane et le MAD (écart absolu médian) ne sont presque pas influencés par les extrêmes, contrairement à la moyenne et à l'écart-type. Ils sont estimés sur le train seulement.

Indice d'erreur de cotation : un **aller-retour**, c'est-à-dire un gros mouvement annulé dès le lendemain.

In [ ]:
aberr = analysis.outliers_report(kaggle, SPLITS, threshold=6)
print("Nombre de jours au-delà de |z| = 6 :", aberr["counts"])
print("Allers-retours suspects :", aberr["spike_reversals"] or "aucun")

# Les rendements extrêmes seulement (les écarts d'ouverture sont expliqués juste après)
top = pd.DataFrame(aberr["top"])
top[top["kind"] == "return"]

In [ ]:
echelle = analysis.RobustScale.fit(r_train)
z = echelle.z(r)
extremes = z[z.abs() > 6]

ax = (r * 100).plot(color=BLEU, lw=0.5, alpha=0.6, title="Rendements extrêmes (|z robuste| > 6)")
ax.scatter(extremes.index, r[extremes.index] * 100, color=ORANGE, zorder=3, s=20)
for d in r.abs().nlargest(4).index:
    ax.annotate(
        d.strftime("%Y-%m-%d"),
        (d, r[d] * 100),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )
plt.show()

> **À retenir :**
> - Les rendements extrêmes sont de **vrais événements** : krach de l'or du 15 avril 2013 (-9,5 %), faillite de Lehman en septembre 2008, Covid en mars 2020. **Aucun aller-retour suspect** n'apparaît, donc pas d'erreur de cotation. Décision : **on les garde**, le modèle doit savoir que ces jours existent.
> - Les nombreux « écarts d'ouverture » signalés sont des faux positifs : dans MT4, l'ouverture est presque toujours égale à la clôture de la veille, donc le moindre écart paraît énorme. Seuls ceux qui suivent les deux trous comptent.

## 8. Volatilité et régimes

La volatilité mesure l'agitation du marché : c'est l'écart-type des rendements sur les 20 derniers jours, annualisé (× √252). On la calcule **uniquement avec le passé** : la valeur du jour t n'utilise que les jours ≤ t.

In [ ]:
reg = analysis.volatility_regimes(r, SPLITS, cfg.features.volatility_window)
vol = analysis.rolling_volatility(r, cfg.features.volatility_window) * 100
seuils = reg["thresholds_from_train"]

ax = vol.plot(
    color=BLEU, lw=0.8, title="Volatilité 20 j annualisée (%) et seuils de régime (train)"
)
for s in seuils.values():
    ax.axhline(s * 100, color=ORANGE, ls="--", lw=1)
plt.show()

bas, haut = seuils["low_mid"], seuils["mid_high"]
print(f"Seuils (train) : basse < {bas:.1%} < moyenne < {haut:.1%} < haute")
pd.DataFrame(reg["regime_share_by_split"]).T.round(2)

In [ ]:
ax = (
    pd.Series(reg["annualized_volatility_by_year"])
    .mul(100)
    .plot.bar(color=BLEU, title="Volatilité annualisée par année (%)")
)
plt.show()

> **À retenir :** le marché alterne des périodes calmes et agitées. La **validation est calme** (peu de jours en volatilité haute) alors que le test l'est moins. Les seuils réglés sur la validation seront donc testés sur un marché plus agité, ce qu'il faudra signaler dans le rapport. Le pic de fin 2025 est **gonflé par le trou** de septembre-octobre 2025.

## 9. Relations entre variables

On cherche ce qui pourrait aider à prédire le rendement de **demain**. On construit quelques variables simples, **toutes connues le soir du jour t**, et on regarde leur lien avec le rendement de t+1.

⚠️ La colonne `r_demain` (rendement de t+1) sert **uniquement à l'analyse**. Elle ne doit jamais devenir une feature du modèle, sinon on prédirait le futur avec le futur.

In [ ]:
f = kaggle.set_index("date")
variables = pd.DataFrame(
    {
        "r": r,  # rendement du jour
        "abs_r": r.abs(),  # taille du mouvement
        "amplitude": (f["high"] - f["low"]) / f["close"],
        "ecart_ouverture": np.log(f["open"] / f["close"].shift(1)),
        "vol_20j": analysis.rolling_volatility(r, 20),
        "r_5j": r.rolling(5).sum(),  # tendance courte
        "volume_ticks": np.log(f["volume"]),
    }
)
variables["r_demain"] = variables["r"].shift(-1)  # CIBLE, analyse seulement
variables["abs_r_demain"] = variables["abs_r"].shift(-1)
train = variables[analysis.assign_split(variables.index, SPLITS).to_numpy() == "train"].dropna()

corr = train.corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title("Corrélations de rang (Spearman), train")
ax.grid(False)
plt.show()

In [ ]:
# Lien direct avec demain : direction (r_demain) et agitation (abs_r_demain)
corr[["r_demain", "abs_r_demain"]].drop(["r_demain", "abs_r_demain"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(train["r"] * 100, train["r_demain"] * 100, s=3, alpha=0.3, color=BLEU)
axes[0].set(
    xlabel="rendement aujourd'hui (%)",
    ylabel="rendement demain (%)",
    title="Direction : pas de lien visible",
)
axes[1].scatter(train["vol_20j"] * 100, train["abs_r_demain"] * 100, s=3, alpha=0.3, color=VERT)
axes[1].set(
    xlabel="volatilité 20 j aujourd'hui (%)",
    ylabel="|rendement| demain (%)",
    title="Agitation : plus la volatilité est haute, plus demain bouge",
)
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(r_train, lags=30, zero=False, ax=axes[0], title="Autocorrélation de r (train)")
plot_acf(r_train.abs(), lags=30, zero=False, ax=axes[1], title="Autocorrélation de |r| (train)")
plt.tight_layout()
plt.show()
acf = analysis.autocorrelation_report(r_train)
print(
    "Ljung-Box (10 retards) p-valeur, r :",
    round(acf["returns"]["ljung_box"][10]["p_value"], 4),
    "| |r| :",
    acf["abs_returns"]["ljung_box"][10]["p_value"],
)

In [ ]:
# Saisonnalité : y a-t-il un « bon » jour de la semaine ou un « bon » mois ? (train)
jours = ["lun", "mar", "mer", "jeu", "ven"]
mois = ["jan", "fév", "mar", "avr", "mai", "jun", "jul", "aoû", "sep", "oct", "nov", "déc"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
par_jour = r_train.groupby(r_train.index.day_of_week).agg(["mean", "sem"]) * 100
axes[0].bar(jours, par_jour["mean"], yerr=1.96 * par_jour["sem"], color=BLEU, capsize=3)
axes[0].set_title("Rendement moyen par jour de la semaine (%, ± IC 95 %)")
par_mois = r_train.groupby(r_train.index.month).agg(["mean", "sem"]) * 100
axes[1].bar(mois, par_mois["mean"], yerr=1.96 * par_mois["sem"], color=BLEU, capsize=3)
axes[1].set_title("Rendement moyen par mois (%, ± IC 95 %)")
plt.tight_layout()
plt.show()

> **À retenir :**
> - **La direction de demain est presque imprévisible** à partir des rendements passés (corrélations proches de 0, Ljung-Box non significatif). La baseline « rendement nul » sera dure à battre : c'est normal sur un marché, et il faudra le dire honnêtement.
> - **L'agitation de demain, elle, est prévisible** : |r| est fortement autocorrélé, et la volatilité d'aujourd'hui annonce celle de demain. → Les features de volatilité (ATR, volatilité 20 j) seront utiles, surtout pour l'agent Risque et pour l'incertitude σ̂.
> - Les barres d'erreur des jours et des mois se chevauchent presque toutes : **pas de saisonnalité nette**. Le jour de la semaine reste une feature possible, mais sans attente forte.

## 10. Stationnarité

Une série est **stationnaire** si son comportement (moyenne, variance) ne dérive pas dans le temps. Les modèles apprennent beaucoup mieux sur une série stationnaire.

- **ADF** : H0 = « non stationnaire ». Une p-valeur < 0,05 indique une série stationnaire.
- **KPSS** : H0 = « stationnaire ». Une p-valeur < 0,05 indique une série non stationnaire.

In [ ]:
log_prix_train = analysis.train_part(np.log(prix), SPLITS)
pd.DataFrame(analysis.stationarity_report(log_prix_train, r_train)).T

> **À retenir :** le **prix n'est pas stationnaire**, contrairement aux **rendements**. On prédit donc le rendement et non le prix : un modèle entraîné sur des prix entre 400 et 1 800 dollars ne saurait pas quoi faire à 4 000 dollars.

## 11. Nettoyage pas à pas

Le nettoyage utilise le module testé `gold_agents.data.clean`, comme la commande `gold-agents clean` : le notebook et le code donnent exactement le même résultat.

| Problème | Décision |
|---|---|
| Doublons de dates | Déjà retirés au chargement (on garde la dernière ligne) |
| Bougies OHLC incohérentes | Aucune dans nos données. Si jamais il y en a : `high`/`low` élargis pour contenir `open` et `close`, ligne marquée `ohlc_fixed` |
| Rendements extrêmes | Vrais événements : **on les garde** |
| Volume (ticks MT4) | Conservé dans le fichier, mais **pas utilisé comme feature fiable** |
| **Trous de 2025-09 et 2026-01** | **Comblés avec Twelve Data** (décision du 28/10/2026) |

### Pourquoi combler avec Twelve Data ?

Twelve Data cote le **même instrument** (XAU/USD au comptant). Avant d'accepter ses barres, on vérifie qu'elles collent à Kaggle sur les ~30 jours ouvrés qui entourent chaque trou (écart des clôtures et corrélation des rendements, même contrôle que `update-data`). Les jours récupérés gardent `source = "twelvedata"` et `filled = True` : on sait toujours d'où vient chaque ligne.

Les barres récupérées sont **figées** dans `data/raw/XAU_1d_gapfill.csv`, avec leur empreinte dans `CHECKSUMS.txt`. Une fois ce fichier créé (par `gold-agents fill-gaps` ou par la cellule ci-dessous), le nettoyage ne dépend plus du réseau et donne toujours le même résultat.

Options écartées, gardées pour comparer :
- `"masquer"` : garder les trous et marquer comme invalide le rendement qui les traverse (`valid_return = False`). C'est honnête, mais on perd environ un mois de test.
- `"couper"` : arrêter le test avant le premier trou. On perd environ 95 jours de test.
- **Interpoler** (tracer une droite entre les deux bords) : écarté, car on inventerait des prix.

Un trou est jugé **anormal** au-delà de `SEUIL_TROU` jours ouvrés manquants d'affilée. Les jours fériés en font 1 ou 2.

In [ ]:
from gold_agents.data import clean as cleaning

STRATEGIE_TROUS = "combler"  # décision retenue ; "masquer" ou "couper" pour comparer
SEUIL_TROU = 3  # jours ouvrés manquants d'affilée au-delà desquels un trou est anormal

trous = cleaning.find_gaps(kaggle, SEUIL_TROU)
pd.DataFrame(trous)

In [ ]:
comblement = None
if STRATEGIE_TROUS == "combler":
    chemin = cfg.data.gapfill_path
    if chemin.exists():
        print(f"Barres déjà figées dans {chemin} : on les relit (pas de réseau).")
    else:
        # Équivalent de `uv run gold-agents fill-gaps` : demande la clé TWELVEDATA_API_KEY.
        from gold_agents.data.sources import fetch_twelvedata
        from gold_agents.data.update import record_checksum

        barres, rapports = cleaning.fetch_gap_bars(kaggle, fetch_twelvedata, min_missing=SEUIL_TROU)
        for f in rapports:
            print(
                f"{f.gap.after} -> {f.gap.before} : {f.n_filled} jour(s) récupéré(s). "
                f"Contrôle : {f.report.describe()}"
            )
        cleaning.write_gapfill(barres, chemin)
        record_checksum(chemin, cfg.data.checksums_path)
    comblement = read_update_csv(chemin)
    source = kaggle
elif STRATEGIE_TROUS == "couper":
    debut_test = pd.Timestamp(SPLITS.test_start).date()
    premier = min((t.after for t in trous if t.after >= debut_test), default=None)
    source = kaggle if premier is None else kaggle[kaggle["date"].dt.date <= premier]
else:
    source = kaggle

propre = cleaning.clean(source, SPLITS, comblement, min_missing=SEUIL_TROU)
print(
    f"Stratégie : {STRATEGIE_TROUS} | lignes : {len(propre)} | comblées : "
    f"{int(propre['filled'].sum())} | bougies corrigées : {int(propre['ohlc_fixed'].sum())} | "
    f"rendements invalides : {int((~propre['valid_return']).sum())}"
)
propre[propre["filled"] | ~propre["valid_return"]][
    ["date", "close", "source", "filled", "log_return", "valid_return", "split"]
]

In [ ]:
# Les trous après comblement : les points Twelve Data (orange) prolongent la courbe Kaggle
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, t in zip(axes, trous[-2:], strict=False):
    debut, fin = (
        pd.Timestamp(t.after) - pd.Timedelta(days=30),
        pd.Timestamp(t.before) + pd.Timedelta(days=30),
    )
    zoom = propre[
        (propre["date"].dt.tz_localize(None) >= debut)
        & (propre["date"].dt.tz_localize(None) <= fin)
    ]
    ax.plot(zoom["date"], zoom["close"], color=GRIS, lw=1)
    k, f = zoom[~zoom["filled"]], zoom[zoom["filled"]]
    ax.scatter(k["date"], k["close"], s=12, color=VERT, label="Kaggle", zorder=3)
    ax.scatter(f["date"], f["close"], s=12, color=ORANGE, label="Twelve Data (comblé)", zorder=3)
    ax.set_title(f"Trou {t.after} -> {t.before}")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
plt.tight_layout()
plt.show()

## 12. Vérifications finales et sauvegarde

Avant d'écrire le fichier, `check_clean` vérifie automatiquement : dates triées et uniques, aucun prix manquant ou négatif, bougies cohérentes, aucun week-end, tous les trous restants marqués, chaque ligne dans un ensemble. Si une règle est violée, le notebook s'arrête et la nomme.

In [ ]:
problemes = cleaning.check_clean(propre, SEUIL_TROU)
assert not problemes, problemes
print("Toutes les vérifications passent ✔")

In [ ]:
# Le résultat : la volatilité recalculée sur la série nettoyée
r_propre = propre.set_index("date")["log_return"].where(propre.set_index("date")["valid_return"])
vol_propre = r_propre.rolling(20, min_periods=15).std() * np.sqrt(252) * 100
fig, ax = plt.subplots()
ax.plot(vol.index, vol, color=ORANGE, lw=1, label="avant nettoyage")
ax.plot(vol_propre.index, vol_propre, color=BLEU, lw=1, label="après nettoyage")
ax.set_title("Volatilité 20 j (%) avant / après le traitement des trous")
ax.legend()
plt.show()

In [ ]:
import json

sortie = cleaning.write_clean(
    propre, cfg.data.processed_dir, cfg.data.raw_path, cfg.data.gapfill_path
)
print(f"Écrit : {sortie}")
json.loads((sortie.parent / "xau_1d_clean.json").read_text(encoding="utf-8"))

## Et ensuite ?

1. Rédiger les constats de ce notebook dans le rapport (base : `docs/constats_eda.md` et la page Notion d'explications).
2. Le même nettoyage se relance sans notebook avec `uv run gold-agents clean`.
3. Phase 2 : features (SMA, EMA, RSI, MACD, ATR, volatilité…) et découpage, sur `data/processed/xau_1d_clean.parquet`.